In [1]:
import pandas as pd
import plotly.graph_objects as go
from nettseval.benchmarks.loader import BenchmarkLoader

benchmark = BenchmarkLoader.load("workers")
source = benchmark.sources["ips"]
print(f"Loaded {source.n_series} series from workers/ips")

Loaded 25 series from workers/ips


In [2]:
def get_full_series(source, ts_id):
    train = source.get_series(ts_id, "train")
    val = source.get_series(ts_id, "val")
    test = source.get_series(ts_id, "test")
    train["split"] = "train"
    val["split"] = "val"
    test["split"] = "test"
    return pd.concat([train, val, test], ignore_index=True)

def sparsity(series: pd.Series) -> float:
    zero_or_nan = (series == 0) | series.isna()
    return zero_or_nan.sum() / len(series)

In [2]:
TARGET = source.target_column

split_colors = {"train": "steelblue", "val": "darkorange", "test": "green"}

for ts_id in source.ts_ids[:10]:
    df = get_full_series(source, ts_id)
    sp = sparsity(df[TARGET])
    test_df = df[df["split"] == "test"]
    sp_test = sparsity(test_df[TARGET])
    zero_pct = (df[TARGET] == 0).sum() / len(df)
    nan_pct = df[TARGET].isna().sum() / len(df)

    print(f"--- id_ip={ts_id} ---")
    print(f"  Total points : {len(df)}")
    print(f"  Sparsity     : {sp:.2%}  (zeros: {zero_pct:.2%}, NaNs: {nan_pct:.2%})")
    print(f"  Test sparsity: {sp_test:.2%}")

    fig = go.Figure()
    for split, color in split_colors.items():
        mask = df["split"] == split
        fig.add_trace(go.Scatter(
            x=df.loc[mask, "datetime"],
            y=df.loc[mask, TARGET],
            mode="lines",
            name=split,
            line=dict(color=color, width=1),
        ))

    fig.update_layout(
        title=f"id_ip={ts_id}  |  sparsity={sp:.2%}  |  test sparsity={sp_test:.2%}",
        xaxis_title="datetime",
        yaxis_title="n_bytes",
        height=300,
        margin=dict(t=40, b=40),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )
    fig.show()

--- id_ip=3846 ---
  Total points : 6718
  Sparsity     : 72.79%  (zeros: 72.79%, NaNs: 0.00%)
  Test sparsity: 81.70%


--- id_ip=21943 ---
  Total points : 6718
  Sparsity     : 76.29%  (zeros: 76.29%, NaNs: 0.00%)
  Test sparsity: 80.95%


--- id_ip=61466 ---
  Total points : 6718
  Sparsity     : 74.41%  (zeros: 74.41%, NaNs: 0.00%)
  Test sparsity: 79.91%


--- id_ip=71574 ---
  Total points : 6718
  Sparsity     : 76.35%  (zeros: 76.35%, NaNs: 0.00%)
  Test sparsity: 78.27%


--- id_ip=142157 ---
  Total points : 6718
  Sparsity     : 72.77%  (zeros: 72.77%, NaNs: 0.00%)
  Test sparsity: 72.32%


--- id_ip=187291 ---
  Total points : 6718
  Sparsity     : 74.56%  (zeros: 74.56%, NaNs: 0.00%)
  Test sparsity: 83.63%


--- id_ip=222483 ---
  Total points : 6718
  Sparsity     : 76.03%  (zeros: 76.03%, NaNs: 0.00%)
  Test sparsity: 77.68%


--- id_ip=258691 ---
  Total points : 6718
  Sparsity     : 67.68%  (zeros: 67.68%, NaNs: 0.00%)
  Test sparsity: 78.94%


--- id_ip=286014 ---
  Total points : 6718
  Sparsity     : 74.83%  (zeros: 74.83%, NaNs: 0.00%)
  Test sparsity: 73.74%


--- id_ip=288481 ---
  Total points : 6718
  Sparsity     : 76.82%  (zeros: 76.82%, NaNs: 0.00%)
  Test sparsity: 79.09%


In [3]:
for ts_id in source.ts_ids:
    df = get_full_series(source, ts_id)
    df["hour"] = df["datetime"].dt.hour
    df["day"] = df["datetime"].dt.dayofweek

    pivot = df.groupby(["day", "hour"])[TARGET].sum().unstack(level="hour")
    pivot.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

    fig = go.Figure(go.Heatmap(
        z=pivot.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        colorscale="YlOrRd",
        colorbar=dict(title="n_bytes"),
    ))
    fig.update_layout(
        title=f"id_ip={ts_id}  |  total traffic by day & hour",
        xaxis_title="Hour of day",
        yaxis_title="Day of week",
        xaxis=dict(tickmode="linear", dtick=1),
        yaxis=dict(autorange="reversed"),
        height=320,
        margin=dict(t=40, b=40),
    )
    fig.show()

In [4]:
rows = []
for ts_id in source.ts_ids:
    df = get_full_series(source, ts_id)
    test_df = df[df["split"] == "test"]
    rows.append({
        "id_ip": ts_id,
        "total_points": len(df),
        "sparsity": sparsity(df[TARGET]),
        "zeros_pct": (df[TARGET] == 0).sum() / len(df),
        "nans_pct": df[TARGET].isna().sum() / len(df),
        "test_sparsity": sparsity(test_df[TARGET]),
    })

summary_df = pd.DataFrame(rows).set_index("id_ip")
avg_row = summary_df.mean().rename("average")
summary_df = pd.concat([summary_df, avg_row.to_frame().T])
summary_df.style.format({
    "sparsity": "{:.2%}",
    "zeros_pct": "{:.2%}",
    "nans_pct": "{:.2%}",
    "test_sparsity": "{:.2%}",
    "total_points": "{:.0f}",
}).background_gradient(subset=["sparsity", "test_sparsity"], cmap="YlOrRd")

,total_points,sparsity,zeros_pct,nans_pct,test_sparsity
3846,6718,72.79%,72.79%,0.00%,81.70%
21943,6718,76.29%,76.29%,0.00%,80.95%
61466,6718,74.41%,74.41%,0.00%,79.91%
71574,6718,76.35%,76.35%,0.00%,78.27%
142157,6718,72.77%,72.77%,0.00%,72.32%
187291,6718,74.56%,74.56%,0.00%,83.63%
222483,6718,76.03%,76.03%,0.00%,77.68%
258691,6718,67.68%,67.68%,0.00%,78.94%
286014,6718,74.83%,74.83%,0.00%,73.74%
288481,6718,76.82%,76.82%,0.00%,79.09%


### 10min bench

In [3]:
benchmark_10min = BenchmarkLoader.load("workers", freq_dir="10min")
source_10min = benchmark_10min.sources["ips"]
TARGET_10min = source_10min.target_column

rows_10min = []
for ts_id in source_10min.ts_ids:
    df = get_full_series(source_10min, ts_id)
    test_df = df[df["split"] == "test"]
    rows_10min.append({
        "id_ip": ts_id,
        "total_points": len(df),
        "sparsity": sparsity(df[TARGET_10min]),
        "zeros_pct": (df[TARGET_10min] == 0).sum() / len(df),
        "nans_pct": df[TARGET_10min].isna().sum() / len(df),
        "test_sparsity": sparsity(test_df[TARGET_10min]),
    })

summary_10min = pd.DataFrame(rows_10min).set_index("id_ip")
avg_row_10min = summary_10min.mean().rename("average")
summary_10min = pd.concat([summary_10min, avg_row_10min.to_frame().T])
summary_10min.style.format({
    "sparsity": "{:.2%}",
    "zeros_pct": "{:.2%}",
    "nans_pct": "{:.2%}",
    "test_sparsity": "{:.2%}",
    "total_points": "{:.0f}",
}).background_gradient(subset=["sparsity", "test_sparsity"], cmap="YlOrRd")

,total_points,sparsity,zeros_pct,nans_pct,test_sparsity
3846,40298,76.52%,76.52%,0.00%,84.50%
21939,40298,71.02%,71.02%,0.00%,87.27%
29898,40298,74.94%,74.94%,0.00%,76.66%
53673,40298,73.33%,73.33%,0.00%,78.76%
111517,40298,76.33%,76.33%,0.00%,80.82%
187291,40298,76.79%,76.79%,0.00%,84.21%
272399,40298,71.70%,71.70%,0.00%,70.57%
303078,40298,72.45%,72.45%,0.00%,84.79%
323771,40298,75.83%,75.83%,0.00%,83.87%
330940,40298,74.99%,74.99%,0.00%,75.31%
